# RelaxedHogbom deconvolution

`RelaxedHogbom` first performs an ordinary Högbom model update. It then jointly adjusts the amplitudes and sub-pixel positions of only the positive point components added by that update. Positions remain inside a local radius expressed in synthesized beams, and the trial is committed only when the measured residual L2 norm improves by the requested fraction.

In [ ]:
import numpy as np

from astroviper.processing_functions.imaging.deconvolution import relaxed_hogbom_clean

size = 32
y, x = np.mgrid[:size, :size]
psf_plane = np.exp(-((x - size // 2) ** 2 + (y - size // 2) ** 2) / 8.0)
psf = np.ascontiguousarray(psf_plane[None, None, None])
truth = np.zeros((size, size))
truth[15, 16] = 0.35
truth[16, 16] = 0.65
dirty = np.fft.ifft2(np.fft.fft2(truth) * np.fft.fft2(psf_plane)).real
dirty = np.roll(dirty, shift=(-size // 2, -size // 2), axis=(0, 1))
residual = np.ascontiguousarray(dirty[None, None, None])
model = np.zeros_like(residual)

result = relaxed_hogbom_clean(
    residual,
    psf,
    model,
    {
        "gain": 0.1,
        "max_iter": 100,
        "relax_constraint_radius": 0.5,
        "relax_max_components": 100,
        "relax_min_improvement": 0.001,
    },
)
result